# ДЗ 13. Файлы и NumPy на заказах магазина 📂🔢
Решите задачи в Google Colab и сдайте **ссылку на блокнот** через LMS (доступ «Все, у кого есть ссылка»).

- **[База]** — обязательно для всех
- **[Челлендж]** — сколько получится
- **[Про]** — для тех, кому легко: работаем с «грязным» `orders_raw.csv`

Каждый ответ — **число + одна фраза**: что это значит для магазина.
Циклы `for` по заказам для расчётов не используем — только массивы NumPy. Цикл можно там, где это сказано в задаче.

**Проверь себя перед сдачей:**
□ Все ячейки запускаются сверху вниз без ошибок (**Среда выполнения → Перезапустить и выполнить все**)
□ У переменных понятные имена
□ Ответы совпадают с «Примером проверки», где он есть
□ JSON-файлы открываются, русские буквы читаются

## [Для всех] Чтение кода — без запуска
Сначала напишите ответ и **почему**, потом запустите и проверьте себя.
```python
# 1
print("25 000".replace(" ", "") + "1")
# 2
print(" ALMATY ".strip().lower())
# 3
a = np.arange(12).reshape(3, 4); print(a[2, 1], a.shape)
# 4
a = np.array([5, 9, 2, 9]); print(a.argmax(), a[1:3])
# 5
print(np.zeros((2, 3)).shape, np.arange(10, 0, -3))
```

Мои предсказания и объяснения:

1.250001 — .replace(" ", "") убирает пробел в "25 000", а + "1" склеивает строки ("25000" и "1"), а не складывает числа.
2.almaty — .strip() убирает пробелы по краям, а .lower() переводит в нижний регистр.
3.9 (3, 4) — матрица $3 \times 4$ (0..11), строка с индексом 2 — [8, 9, 10, 11], элемент [2, 1] равен 9, форма (3, 4).   
4.1 [9 2] — a.argmax() даёт индекс первого максимума (9 стоит на индексе 1), срез a[1:3] берёт элементы с индексами 1 и 2.
5.(2, 3) [10  7  4  1] — форма массива нулей (2, 3), а np.arange(10, 0, -3) идёт от 10 вниз с шагом -3, пока значение больше

In [4]:
import numpy as np

# 1
print("25 000".replace(" ", "") + "1")
# 2
print(" ALMATY ".strip().lower())
# 3
a = np.arange(12).reshape(3, 4)
print(a[2, 1], a.shape)
# 4
a = np.array([5, 9, 2, 9])
print(a.argmax(), a[1:3])
# 5
print(np.zeros((2, 3)).shape, np.arange(10, 0, -3))

250001
almaty
9 (3, 4)
1 [9 2]
(2, 3) [10  7  4  1]


## Загрузка данных

In [27]:
# Загрузка: в Colab нажмите «Выбрать файлы» и укажите orders_1.csv (и orders_raw.csv для уровня Про) из LMS.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders_1.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))
print("orders_raw.csv на месте:", os.path.exists("orders_raw.csv"))

orders.csv на месте: True
orders_raw.csv на месте: True


In [28]:
import csv
import json
import numpy as np

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

print(len(rows))
print(rows[0])

500
{'order_id': 'ORD-0001', 'date': '2026-03-01', 'customer_id': 'C065', 'city': 'Актобе', 'category': 'Смартфоны', 'product': 'Xiaomi Redmi Note 13', 'price': '99000', 'qty': '2', 'status': 'оплачен'}


## [База]

### Задача 1. Чистим одну строку
Строка из `orders_raw.csv`:
```
ORD-0006,2026-03-04,C091,ALMATY,Ноутбуки,ASUS VivoBook 15,319000 ₸,1, оплачен
```
Достаньте из неё:
- город — маленькими буквами, без пробелов;
- цену — как число `int`, без `₸` и пробелов;
- статус — без пробелов по краям.

Проверьте: `статус == "оплачен"` должно дать `True`.

**Пример проверки:** `almaty`, `319000`, `True`.

In [29]:
row = "ORD-0006,2026-03-04,C091,ALMATY,Ноутбуки,ASUS VivoBook 15,319000 ₸,1, оплачен"

parts = row.split(",")

city = parts[3].strip().lower()
price = int(parts[6].replace("₸", "").replace(" ", "").strip())
status = parts[8].strip()

print(city, price, status == "оплачен")

almaty 319000 True


### Задача 2. TXT-файл с товарами
Запишите в `products.txt` названия товаров из **первых 5 заказов** (`rows`), по одному в строке.
Прочитайте файл обратно в список **без** `"\n"` и без пустой строки в конце.

**Пример проверки:** список из 5 строк, первая — `'Xiaomi Redmi Note 13'`.

In [30]:
first_5_products = [row["product"].strip() for row in rows[:5]]

with open("products.txt", "w", encoding="utf-8") as f:
    for product in first_5_products:
        f.write(product + "\n")

with open("products.txt", "r", encoding="utf-8") as f:
    products_list = [line.strip() for line in f if line.strip()]

print(len(products_list), products_list)

5 ['Xiaomi Redmi Note 13', 'Клавиатура Logitech K120', 'Samsung Galaxy Tab A9', 'Samsung Galaxy A55', 'Монитор LG 27 дюймов']


### Задача 3. Массив и `reshape`
Создайте массив чисел от 1 до 24 и разложите его в таблицу **4 строки × 6 столбцов**. Выведите:
1. число в 3-й строке, 2-м столбце (помните: счёт с нуля!);
2. последний столбец;
3. сумму первой строки;
4. форму таблицы, если те же числа разложить в 6 строк.

**Пример проверки:** `14`, `[ 6 12 18 24]`, `21`, `(6, 4)`.

In [32]:
import numpy as np
arr = np.arange(1, 25).reshape(4, 6)
elem_3_2 = arr[2, 1]
last_col = arr[:, -1]
first_row_sum = arr[0].sum()
shape_6_rows = np.arange(1, 25).reshape(6, -1).shape
print(elem_3_2)
print(last_col)
print(first_row_sum)
print(shape_6_rows)

14
[ 6 12 18 24]
21
(6, 4)


### Задача 4. Заказы в NumPy
Соберите из `rows` массивы `price`, `qty`, `product` (цикл для сбора списков можно, как на уроке). Посчитайте `revenue = price * qty`.
1. Сколько всего заказов?
2. Самый **дешёвый** заказ: сумма и товар (`argmin`).
3. Самый дорогой заказ: сумма. Во сколько раз он дороже самого дешёвого?

**Пример проверки:** `500` заказов; самый дешёвый — `7 500 ₸`, `Мышь Logitech M185`; самый дорогой — `1 098 000 ₸`.

In [33]:
import numpy as np

price = np.array([int(r["price"]) for r in rows])
qty = np.array([int(r["qty"]) for r in rows])
product = np.array([r["product"].strip() for r in rows])

revenue = price * qty

total_orders = len(revenue)
min_idx = revenue.argmin()
max_idx = revenue.argmax()

min_rev = revenue[min_idx]
min_prod = product[min_idx]
max_rev = revenue[max_idx]
ratio = max_rev / min_rev

print(f"1. {total_orders} — столько всего заказов зафиксировано в выгрузке магазина.")
print(
    f"2. {min_rev} — сумма самого дешёвого заказа ({min_prod}), что задаёт нижнюю границу чека."
)
print(
    f"3. {max_rev} — сумма самого дорогого заказа, она в {ratio:.2f} раза больше самого дешёвого чека."
)

1. 500 — столько всего заказов зафиксировано в выгрузке магазина.
2. 7500 — сумма самого дешёвого заказа (Мышь Logitech M185), что задаёт нижнюю границу чека.
3. 1098000 — сумма самого дорогого заказа, она в 146.40 раза больше самого дешёвого чека.


### Задача 5. Отчёт в JSON
Сохраните результаты задачи 4 в `hw_base.json`:
```json
{
  "заказов": 500,
  "самый_дешёвый": {"сумма": 7500, "товар": "Мышь Logitech M185"},
  "самый_дорогой": 1098000
}
```
Прочитайте файл через `json.load` и выведите товар самого дешёвого заказа.
Не забудьте: числа NumPy → `int()`, текст NumPy → `str()`.

In [35]:
import json
report = {
    "заказов": int(total_orders),
    "самый_дешёвый": {
        "сумма": int(min_rev),
        "товар": str(min_prod),
    },
    "самый_дорогой": int(max_rev),
}
with open("hw_base.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)
with open("hw_base.json", "r", encoding="utf-8") as f:
    loaded_report = json.load(f)
print(loaded_report["самый_дешёвый"]["товар"])

Мышь Logitech M185


## [Челлендж]

### Задача 6. Сколько потеряли на отменах
Соберите массив `status`. Посчитайте:
- сколько заказов отменено;
- какая это доля от всех заказов;
- на какую сумму были отменённые заказы.

Подсказка: `status == "отменён"` — массив `True/False`; `.sum()` считает `True`, `.mean()` даёт долю.

**Пример проверки:** `45` заказов, `9%`, `8 224 000 ₸`.

In [36]:
import numpy as np

status = np.array([r["status"].strip() for r in rows])
cancelled_mask = status == "отменён"

cancelled_count = int(cancelled_mask.sum())
cancelled_share_pct = cancelled_mask.mean() * 100
cancelled_sum = int(revenue[cancelled_mask].sum())

print(f"{cancelled_count} — столько заказов было отменено в выгрузке.")
print(
    f"{cancelled_share_pct:.0f}% — такая доля всех заказов приходится на отмены."
)
print(
    f"{cancelled_sum} — на такую сумму были отменённые заказы (потенциальная упущенная выручка)."
)

45 — столько заказов было отменено в выгрузке.
9% — такая доля всех заказов приходится на отмены.
8224000 — на такую сумму были отменённые заказы (потенциальная упущенная выручка).


### Задача 7. Неделя курьера
Курьер развозит по 5 заказов в день. Возьмите первые **35** заказов и разложите в таблицу «дни × заказы».
1. Какую форму передать в `reshape`?
2. Посчитайте сумму за каждый день — список из 7 чисел (цикл по строкам таблицы можно).
3. В какой день сумма была самой большой?

**Пример проверки:** самый «дорогой» день — **7-й**, `1 227 000 ₸`.

In [37]:
import numpy as np

week_revenue = revenue[:35].reshape(7, 5)
daily_sums = [int(day_row.sum()) for day_row in week_revenue]
best_day_idx = int(np.argmax(daily_sums))
best_day_num = best_day_idx + 1
best_day_sum = daily_sums[best_day_idx]

print("1. (7, 5) — такую форму нужно передать в reshape для таблицы «7 дней × 5 заказов».")
print(f"2. {daily_sums} — суммы выручки курьера по каждому из 7 дней.")
print(
    f"3. {best_day_num}-й день — в этот день сумма была самой большой и составила {best_day_sum} ₸."
)

1. (7, 5) — такую форму нужно передать в reshape для таблицы «7 дней × 5 заказов».
2. [699000, 925000, 1094000, 636000, 1047000, 571000, 1227000] — суммы выручки курьера по каждому из 7 дней.
3. 7-й день — в этот день сумма была самой большой и составила 1227000 ₸.


### Задача 8. Средний чек по городам
Соберите массив `city`. Для **каждого** города из `np.unique(city)` посчитайте число заказов и средний чек
(цикл по городам можно, внутри — маски). Сохраните в `cities.json`:
```json
{
  "Актобе": {"заказов": 33, "средний_чек": 137136},
  "Алматы": {"заказов": 133, "средний_чек": 193011},
  ...
}
```
В каком городе самый высокий средний чек? Почему в Атырау средний чек считать рискованно?

**Пример проверки:** самый высокий средний чек — **Караганда**, ≈ `208 826 ₸`.

In [38]:
import json
import numpy as np

city = np.array([r["city"].strip() for r in rows])

cities_report = {}
for c in np.unique(city):
    mask = city == c
    orders_cnt = int(mask.sum())
    avg_check = int(round(float(revenue[mask].mean())))
    cities_report[str(c)] = {
        "заказов": orders_cnt,
        "средний_чек": avg_check,
    }

with open("cities.json", "w", encoding="utf-8") as f:
    json.dump(cities_report, f, ensure_ascii=False, indent=2)

best_city = max(cities_report.items(), key=lambda x: x[1]["средний_чек"])
atyrau_orders = cities_report.get("Атырау", {}).get("заказов", 0)

print(
    f"{best_city[1]['средний_чек']} — самый высокий средний чек зафиксирован в городе {best_city[0]}."
)
print(
    f"{atyrau_orders} — столько заказов в Атырау, поэтому считать средний чек по такой выборке рискованно из-за малого числа наблюдений и возможного влияния единичных дорогих покупок."
)

208826 — самый высокий средний чек зафиксирован в городе Караганда.
2 — столько заказов в Атырау, поэтому считать средний чек по такой выборке рискованно из-за малого числа наблюдений и возможного влияния единичных дорогих покупок.


## [Про] «Грязный» `orders_raw.csv`
Это те же заказы, но «как пришли из системы»: цены с `₸` и пробелами, города латиницей и в разном регистре, дубли строк, мусор вместо чисел.
Задача — **измерить грязь** и сохранить отчёт. NumPy пригодится, но сначала строки надо почистить.

In [ ]:
with open("orders_raw.csv", encoding="utf-8") as f:
    raw = list(csv.DictReader(f))
print(len(raw))
print(raw[0])

### Задача 9. Дубли
Сколько строк в `raw`? Сколько **уникальных** `order_id`? Сколько строк — лишние дубли?
Подсказка: `np.unique(массив)` возвращает массив без повторов.

**Пример проверки:** `518` строк, `500` уникальных заказов, `18` дублей.

In [ ]:
# TODO

### Задача 10. Функция `clean_price`
Напишите функцию `clean_price(s)`, которая:
- убирает `₸` и все пробелы;
- переводит в `float` (`"12000.0"` → `12000.0`);
- если перевести нельзя (`"нет данных"`, `"abc"`, пустая строка) — возвращает `None`.

```python
def clean_price(s):
    s = ...                     # чистим
    try:
        return float(s)
    except ValueError:          # float("abc") вызывает ValueError
        return None
```
Проверьте на `"99000 ₸"`, `"25 000"`, `"12000.0"`, `"abc"`. Сколько строк в `raw` с ценой, которую нельзя прочитать?

**Пример проверки:** `99000.0`, `25000.0`, `12000.0`, `None`; плохих цен — `10`.

In [ ]:
# TODO

### Задача 11. Города и статусы
1. Сколько **разных** написаний города в `raw`? (`np.unique` + `len`)
2. Приведите города к одному виду: `strip()` → `lower()` → словарь `CITY` (допишите в него латиницу: `pavlodar`, `aktobe`, `karaganda`, …). Сколько городов осталось?
3. То же со статусами: сколько вариантов было и сколько стало после `strip().lower()`?

**Пример проверки:** города `25` → `7`; статусы `8` → `2`.

In [ ]:
CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент"}
# TODO: допишите словарь и посчитайте

### Задача 12. Отчёт о качестве данных
Сохраните всё найденное в `raw_report.json`:
```json
{
  "строк": 518,
  "уникальных_заказов": 500,
  "дублей": 18,
  "плохих_цен": 10,
  "написаний_города": {"было": 25, "стало": 7},
  "вариантов_статуса": {"было": 8, "стало": 2}
}
```
И ответьте 2–3 предложениями: **что случится, если посчитать выручку по `orders_raw.csv` без очистки?** Назовите хотя бы две проблемы.

In [ ]:
# TODO

Мой ответ: